# **Projeto - Assistente de IA para Companhia Aérea**

Neste notebook de estudo, vamos juntar os conceitos aprendidos anteriormente para construir na prática um **Assistente de Suporte ao Cliente** para uma companhia aérea fictícia (Ramon Flight).

### **Objetivos de Aprendizagem:**
Ao longo desta prática, você irá explorar e aplicar três conceitos fundamentais:

1. **Integração de LLM:** Como configurar o cliente da API para se comunicar com modelos de linguagem avançados.
2. **Interface Visual com Gradio:** Como construir rapidamente uma interface de chat interativa (`gr.ChatInterface`) para que o usuário possa conversar com a IA.
3. **Function Calling (Chamada de Ferramentas):** Aprender como dar ao modelo a habilidade de interagir com o mundo real e com sistemas externos (como bancos de dados). O modelo não apenas gera texto, mas sabe *quando* e *como* acionar funções Python locais para buscar informações precisas (ex: consultar preços de passagens).

---


In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Inicialização

load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

if not api_key:
    print("Nenhuma chave de API da OpenRouter foi encontrada - por favor verifique o seu arquivo .env!")
elif not api_key.startswith("sk-or-"):
    print("Uma chave de API foi encontrada, mas não começa com sk-or-; por favor, verifique se você está usando a chave correta da OpenRouter")
else:
    print("Chave de API da OpenRouter encontrada e parece boa até agora!")

openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)
MODEL = 'nvidia/nemotron-3-super-120b-a12b:free'


Chave de API da OpenRouter encontrada e parece boa até agora!


In [3]:
system_message = """
Você é um assistente útil de uma companhia aérea chamada Ramon Flight.
Dê respostas curtas e corteses, com não mais de 1 frase.
Seja sempre preciso. Se não souber a resposta, diga que não sabe.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Entendendo as Ferramentas (Function Calling)

As **Ferramentas (Tools)** são um recurso incrivelmente poderoso fornecido pelos LLMs modernos (como os modelos da OpenAI, Anthropic e outros disponíveis no OpenRouter).

O fluxo padrão de um LLM é receber texto e gerar texto. Mas com as *ferramentas*, você pode:
1. **Escrever uma função local** em Python (ex: consultar o banco de dados de passagens).
2. **Descrever essa função** para o LLM no formato JSON esperado pela API.
3. Permitir que o LLM, em vez de responder diretamente ao usuário, **retorne um pedido estruturado** solicitando que você execute aquela função para ele.


In [5]:
# Vamos começar fazendo uma função útil

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Ferramenta chamada para a cidade {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Preço de passagem desconhecido")
    return f"O preço de uma passagem para {destination_city} é {price}"


In [6]:
get_ticket_price("London")

Ferramenta chamada para a cidade London


'O preço de uma passagem para London é $799'

In [7]:
# Há uma estrutura de dicionário específica que é necessária para descrever nossa função:

price_function = {
    "name": "get_ticket_price",
    "description": "Obter o preço de uma passagem de ida e volta para a cidade de destino.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "A cidade para a qual o cliente deseja viajar",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [8]:
# E isso é incluído em uma lista de ferramentas:

tools = [{"type": "function", "function": price_function}]

In [9]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Obter o preço de uma passagem de ida e volta para a cidade de destino.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'A cidade para a qual o cliente deseja viajar'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Fazendo o LLM usar nossa ferramenta

Agora que temos nossa função Python e um "dicionário" descrevendo-a, precisamos avisar o modelo de que essa ferramenta existe.

Há alguns detalhes importantes no fluxo da API para permitir essa comunicação:
- Em vez de responder com o texto final, a API pode parar a geração e retornar um motivo de parada (`finish_reason`) igual a `"tool_calls"`.
- O que nós fazemos no nosso código é capturar essa intenção, **executar a função Python localmente** extraindo os argumentos enviados pela IA e, em seguida, devolver o resultado da função de volta como uma nova mensagem na conversa.

Dessa forma, o modelo usa o resultado que extraímos do banco para gerar a resposta em linguagem natural para o usuário. Veja como fica a nova estrutura da função de chat:

In [10]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [11]:
# Temos que escrever a função handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Ferramenta chamada para a cidade Londres
Ferramenta chamada para a cidade London


## Evoluindo nosso Assistente: Múltiplas Chamadas

Nossa primeira versão funciona perfeitamente para consultas simples. No entanto, em um cenário real, o usuário pode fazer pedidos mais complexos que exigem o uso da ferramenta diversas vezes, como: *"Quero saber o preço das passagens para Londres e também para Paris"*.

Para cobrir todos os cenários, vamos implementar duas melhorias fundamentais no nosso código:

1. **Lidar com chamadas paralelas:** O modelo pode pedir para rodar a função `get_ticket_price` para duas cidades diferentes *ao mesmo tempo* na mesma resposta.
2. **Lidar com chamadas sequenciais:** O modelo pode chamar uma ferramenta, analisar a resposta recebida e, com base nisso, decidir que precisa chamar a ferramenta novamente antes de dar a resposta final ao usuário.

Vamos aprimorar nosso assistente para suportar esse fluxo contínuo:

In [13]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [14]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [15]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Ferramenta chamada para a cidade London


In [16]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [17]:
import sqlite3


In [18]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [19]:
def get_ticket_price(city):
    print(f"FERRAMENTA DE BANCO DE DADOS CHAMADA: Obtendo preço para {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"O preço da passagem para {city} é ${result[0]}" if result else "Nenhum dado de preço disponível para esta cidade"

In [20]:
get_ticket_price("London")

FERRAMENTA DE BANCO DE DADOS CHAMADA: Obtendo preço para London


'Nenhum dado de preço disponível para esta cidade'

In [21]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [22]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [23]:
get_ticket_price("Tokyo")

FERRAMENTA DE BANCO DE DADOS CHAMADA: Obtendo preço para Tokyo


'O preço da passagem para Tokyo é $1420.0'

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


FERRAMENTA DE BANCO DE DADOS CHAMADA: Obtendo preço para London
FERRAMENTA DE BANCO DE DADOS CHAMADA: Obtendo preço para Paris
